# 06 Molecular-Space Visualization: From High-Dimensional Fingerprints to a 2D Map

This section projects fingerprints from high-dimensional space into two dimensions to help inspect molecular space.

Application scenarios include checking data coverage, finding outliers, observing property gradients, and building intuition for active learning or experimental design. This section uses scikit-learn PCA as a basic linear dimensionality-reduction method. It is useful for teaching, but it does not replace rigorous similarity or extrapolation evaluation.

## Intuition

Each molecule can be viewed as a point in fingerprint space. Similar molecules are often closer, but a 2D map is only a projection and loses information.

Chemically, the map helps ask: do hydrophobic molecules cluster? Do high-molecular-weight molecules occupy a region? Does logS vary smoothly? If no clear region appears, it does not mean the model is useless; the 2D projection may simply be insufficient for high-dimensional structure.

## Mathematical and Chemical Definitions

PCA finds directions of maximum variance and projects a high-dimensional matrix `X` onto a few principal components:

```text
X_high_dim -> PC1, PC2
```

PCA does not know chemistry. It only uses numerical variance.

## Prepare PCA Input

This cell samples ESOL molecules, computes Morgan fingerprints, and compresses them to two dimensions with PCA. `explained_variance_ratio_` reports how much numerical variance PC1/PC2 retain; it is usually not very high because fingerprints are high-dimensional and sparse.

In [ ]:
from pathlib import Path

START = Path.cwd().resolve()
for candidate in [START, *START.parents]:
    if (candidate / "data").exists() and (candidate / "notebooks").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Cannot find the materials root. Start Jupyter from the materials directory "
        "or from one of its subdirectories."
    )

DATA = ROOT / "data"
RAW = DATA / "raw"
EXAMPLES = DATA / "examples"
RANDOM_STATE = 42

print("materials root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem, DataStructs
from rdkit.Chem import Crippen, Descriptors, Draw, Lipinski, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

sns.set_theme(style="whitegrid", context="notebook")


def mol_from_smiles(smiles):
    if pd.isna(smiles):
        return None
    return Chem.MolFromSmiles(str(smiles).strip())


def canonical_smiles(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)


def scaffold_smiles(mol):
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold)


def descriptor_record(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return {
        "MolWt": Descriptors.MolWt(mol),
        "MolLogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "AromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
        "canonical_smiles": canonical_smiles(smiles),
        "scaffold": scaffold_smiles(mol),
    }


def build_esol_features():
    raw = pd.read_csv(RAW / "esol.csv")
    rows = []
    for row_id, row in raw.reset_index(drop=True).iterrows():
        desc = descriptor_record(row["smiles"])
        if desc is None:
            continue
        desc.update(
            {
                "row_id": row_id,
                "smiles": str(row["smiles"]).strip(),
                "logS": float(row["log solubility (mol/L)"]),
            }
        )
        rows.append(desc)
    return pd.DataFrame(rows)


def fingerprint_array(smiles, n_bits=1024):
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=n_bits)
    matrix = np.zeros((len(smiles), n_bits), dtype=np.int8)
    for idx, smi in enumerate(smiles):
        fp = generator.GetFingerprint(mol_from_smiles(smi))
        DataStructs.ConvertToNumpyArray(fp, matrix[idx])
    return matrix


DESCRIPTOR_COLUMNS = [
    "MolWt",
    "MolLogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds",
    "RingCount",
    "AromaticRings",
    "FractionCSP3",
    "HeavyAtomCount",
]

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

esol = build_esol_features()

# For speed, sample at most 600 molecules; random_state makes the result reproducible.
sample = esol.sample(n=min(600, len(esol)), random_state=RANDOM_STATE).reset_index(drop=True)
fp = fingerprint_array(sample["smiles"], n_bits=384)

# PCA only sees numerical variance in the fingerprint matrix; it does not know logS or chemical mechanisms.
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(fp)

space = sample[["smiles", "logS", "MolWt", "MolLogP", "TPSA"]].copy()
space["PC1"] = coords[:, 0]
space["PC2"] = coords[:, 1]
print("explained variance ratio:", pca.explained_variance_ratio_.round(3))
space.head()

## Code

The next plot uses color to show logS. Inspect whether spatial position relates to solubility, molecular weight, or LogP.

In [ ]:
# If colors form continuous regions, the 2D projection captures some structure differences related to logS.
plt.figure(figsize=(7, 5))
scatter = plt.scatter(space["PC1"], space["PC2"], c=space["logS"], cmap="coolwarm", s=24, alpha=0.8)
plt.colorbar(scatter, label="logS")
plt.title("PCA of Morgan fingerprints")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()

## Recolor Points with a Descriptor

The same 2D coordinates can be colored by different chemical quantities.

Separate two concepts: "position is determined by the fingerprint", while "color is an overlaid property used for interpretation".

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=space, x="PC1", y="PC2", hue="MolWt", palette="viridis", ax=axes[0], legend=False)
axes[0].set_title("Colored by MolWt")
sns.scatterplot(data=space, x="PC1", y="PC2", hue="MolLogP", palette="magma", ax=axes[1], legend=False)
axes[1].set_title("Colored by MolLogP")
plt.tight_layout()

## Inspect Molecules at the Projection Edges

This cell extracts molecules at extreme PC1/PC2 positions. Edge points often help explain what structural differences a PCA direction may reflect, but do not interpret PC1/PC2 as a single chemical property.

In [ ]:
# Select molecules at both ends of PC1/PC2 to inspect what structures appear near the plot edges.
extreme = pd.concat(
    [
        space.nsmallest(3, "PC1"),
        space.nlargest(3, "PC1"),
        space.nsmallest(3, "PC2"),
        space.nlargest(3, "PC2"),
    ]
).drop_duplicates().head(12)

Draw.MolsToGridImage(
    [mol_from_smiles(s) for s in extreme["smiles"]],
    legends=[f"logS={v:.1f}" for v in extreme["logS"]],
    molsPerRow=4,
    subImgSize=(230, 170),
)

## Observation Questions

1. Are neighboring molecules on the PCA plot necessarily chemically similar?
2. Does logS color form a clear region?
3. What information is lost when a high-dimensional fingerprint is projected to two dimensions?

### Hints

1. Not necessarily. 2D proximity only means closeness along PC1/PC2; high-dimensional fingerprint differences may be compressed away.
2. A color gradient suggests that some structural differences relate to logS. Mixed colors may mean logS depends on many factors or the projection is insufficient.
3. Local structure bits, rare fragments, nonlinear relationships, and high-dimensional neighbor relations can be lost. A 2D plot is good for asking questions, not for final evidence.

## Summary

Molecular-space maps are useful for intuition, outlier detection, and storytelling, but they do not replace rigorous evaluation. 2D distance is only one compressed view of high-dimensional structure.